# les bibliothèques

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Encodeur

Nous allons définir :

Chaque jeton d'entrée est converti en un vecteur dense (embedding).

Le GRU traite la séquence jeton par jeton, en mettant à jour son état caché.

L'état caché final est renvoyé sous forme de vecteur de contexte, résumant la séquence d'entrée.

In [2]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hidden_dim)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, hidden = self.rnn(embedded)
        return hidden

# Décodeur

Nous allons définir le décodeur :

Prend le jeton d'entrée actuel et le convertit en un vecteur d'intégration.

GRU utilise l'état caché précédent (ou vecteur de contexte initialement) pour calculer le nouvel état caché.

Le résultat est traité par une couche linéaire pour obtenir les probabilités prédites des jetons.

In [3]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, input, hidden):
        input = input.unsqueeze(0)
        embedded = self.embedding(input)
        output, hidden = self.rnn(embedded, hidden)
        prediction = self.fc(output.squeeze(0))
        return prediction, hidden

# Modèle Seq2Seq 

In [4]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg=None, max_len=10, teacher_forcing_ratio=0.5):
        batch_size = src.shape[1]
        trg_vocab_size = self.decoder.fc.out_features
        outputs = []

        hidden = self.encoder(src)

        input = torch.zeros(batch_size, dtype=torch.long).to(self.device)

        for t in range(max_len):
            output, hidden = self.decoder(input, hidden)
            top1 = output.argmax(1)
            outputs.append(top1.unsqueeze(0))

            if trg is not None and t < trg.shape[0] and torch.rand(1).item() < teacher_forcing_ratio:
                input = trg[t]
            else:
                input = top1

        outputs = torch.cat(outputs, dim=0)
        return outputs

# Utilisation

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VOCAB_SIZE = 10
EMB_DIM = 8
HID_DIM = 16
SEQ_LEN = 5
BATCH_SIZE = 2

enc = Encoder(VOCAB_SIZE, EMB_DIM, HID_DIM)
dec = Decoder(VOCAB_SIZE, EMB_DIM, HID_DIM)
model = Seq2Seq(enc, dec, device).to(device)

src = torch.randint(1, VOCAB_SIZE, (SEQ_LEN, BATCH_SIZE)).to(device)
trg = torch.randint(1, VOCAB_SIZE, (SEQ_LEN, BATCH_SIZE)).to(device)

outputs = model(src, trg, max_len=SEQ_LEN, teacher_forcing_ratio=0.7)

print("Source sequence (input tokens):")
print(src.T)
print("\nTarget sequence (true tokens):")
print(trg.T)
print("\nPredicted sequence (model output tokens):")
print(outputs.T)

Source sequence (input tokens):
tensor([[5, 3, 8, 8, 1],
        [6, 9, 3, 1, 2]])

Target sequence (true tokens):
tensor([[6, 6, 6, 9, 2],
        [2, 8, 2, 2, 5]])

Predicted sequence (model output tokens):
tensor([[6, 3, 6, 6, 3],
        [6, 3, 6, 8, 8]])


Уважаемые коллеги!
Поздравляю Вас с Наступающим Новым  2026 Годом!!!!
Пусть Новый год принесет Вам, Вашим близким и родным
Благополучие, счастье и радостные события!
Пусть грядущий год будет полон грандиозных приключений и новых возможностей!!! 
 
Счастливого Нового Года!!! 
